# NB_Purview_Copilot_BronzeToSilver
**Source:** `AI_Medallion.Bronze.purview_copilot_usage`  
**Targets (reads):** `okta_user_for_ai`, `hc_silver_historical`  
**Target:** `AI_Medallion.Silver.purview_copilot_usage` (incremental merge on `purview_copilot_usage_record_id`)  
**Run after:** `NB_OktaUserforAI_Bronze_To_Silver`, `NB_HCHistorical_Bronze_To_Silver`  
**Filter:** `creation_date >= MEDALLION_PURVIEW_MIN_CREATION_DATE` (default `2026-10-01`; set `2026-09-01` to test)

In [ ]:
# Workspace folder: C:\spark-dev\CoE_transformation_framework (see README)
import os
from pathlib import Path
_cwd = Path.cwd()
if not (_cwd / "Common" / "bootstrap.py").is_file():
    _repo = _cwd.parent
    if (_repo / "Common" / "bootstrap.py").is_file():
        os.chdir(_repo)
os.environ.setdefault("MEDALLION_DEBUG_READ", "1")
os.environ.setdefault("MEDALLION_LOCAL_TABLE_CACHE", "1")
os.environ.setdefault("MEDALLION_SCAN_CHUNK_ROWS", "75000")
os.environ.setdefault("MEDALLION_SCAN_RETRIES", "6")
# Test window: os.environ["MEDALLION_PURVIEW_MIN_CREATION_DATE"] = "2026-09-01"
os.environ.setdefault("MEDALLION_PURVIEW_MIN_CREATION_DATE", "2026-10-01")


In [ ]:
%run ./Common/bootstrap


In [ ]:
bronze = read_table_broken_lineage(TABLE_PURVIEW_COPILOT_BRONZE, "purview_copilot_bronze")


In [ ]:
okta = read_table_broken_lineage(TABLE_OKTA_SILVER, "okta_silver")


In [ ]:
hc = read_table_broken_lineage(TABLE_HC_SILVER, "hc_silver")


In [ ]:
apply_local_merge_spark_conf(spark)

min_date = os.environ.get(
    "MEDALLION_PURVIEW_MIN_CREATION_DATE", PURVIEW_COPILOT_MIN_CREATION_DATE
)
print(f"purview_copilot_usage: creation_date >= {min_date}")

write_df = build_purview_copilot_usage_silver(
    bronze,
    okta,
    hc,
    min_creation_date=min_date,
)

write_df = break_lineage_local(write_df, "purview_copilot_silver")
publish_merge_staging(write_df, TABLE_PURVIEW_COPILOT_SILVER)


In [ ]:
merge_incremental(
    spark,
    write_df,
    TABLE_PURVIEW_COPILOT_SILVER,
    PURVIEW_COPILOT_MERGE_KEY,
)
